# Notebook 15 — 3B Ceiling Baseline · ASDiv
## SLM-to-SLM Guided Reasoning — Compute Ceiling Comparison

**Purpose:** Run Qwen2.5-3B alone with 5 votes — no fine-tuning, no LoRA, no guide.
This is the **compute ceiling** (15B param-passes) vs our pipeline's 10.5B.

| Condition | Setup | Compute |
|-----------|-------|---------|
| Baseline | 1.5B × 5 | 7.5B |
| **Our Pipeline** | 3B guide (LoRA) + 1.5B × 5 | **10.5B** |
| **← This notebook** | 3B base × 5 | **15.0B** |

Same 300 ASDiv questions · Same seed=42 · Same temp=0.4 · All 4 angles including per-operation breakdown

In [1]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [2]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print('HuggingFace login done')

HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU check
import os, json, re, time
import torch
import numpy as np
from collections import Counter, defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/asdiv_ceiling"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"PyTorch : {torch.__version__}")
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {props.total_memory/1024**3:.1f} GB")
else:
    print("No GPU detected")
print(f"Output  : {OUTPUT_DIR}")

PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 15.9 GB
Output  : /kaggle/working/asdiv_ceiling


In [4]:
# CELL 4 -- Configuration
# ONE model only. No guide. No LoRA. No fine-tuning.
# Ceiling condition: 3B x 5 votes = 15B param-passes.
CONFIG = {
    "model_name"          : "Qwen/Qwen2.5-3B-Instruct",
    "model_params_B"      : 3.0,
    # Dataset
    "dataset_name"        : "EleutherAI/asdiv",
    "dataset_split"       : "validation",     # ASDiv uses validation split
    "max_eval_samples"    : 300,              # same as pipeline N=300 run
    "random_seed"         : 42,              # FIXED -- must match pipeline run
    # Voting
    "n_votes"             : 5,
    "vote_temperature"    : 0.4,             # same as pipeline N=300 run
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 350,
    # Known results from N=300 pipeline run (for comparison in Angle 1)
    "known_baseline_acc"  : 53.7,            # 1.5B x 5 @ 7.5B
    "known_pipeline_acc"  : 64.0,            # 3B+LoRA + 1.5B x 5 @ 10.5B
    "known_baseline_B"    : 7.5,
    "known_pipeline_B"    : 10.5,
    # Known per-operation pipeline results (from N=300 run, for Angle 4)
    "known_pipeline_by_op": {
        "Addition"      : {"n": 61,  "guided": 73.8, "baseline": 55.7},
        "Division"      : {"n": 40,  "guided": 60.0, "baseline": 40.0},
        "Multi-step"    : {"n": 89,  "guided": 53.9, "baseline": 53.9},
        "Multiplication": {"n": 43,  "guided": 72.1, "baseline": 55.8},
        "Subtraction"   : {"n": 67,  "guided": 64.2, "baseline": 56.7},
    },
    # Output paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "angle4_file"         : f"{OUTPUT_DIR}/angle4_by_operation_type.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}
print("Config ready:")
for k, v in CONFIG.items():
    if k != "known_pipeline_by_op":
        print(f"  {k:<26}: {v}")

Config ready:
  model_name                : Qwen/Qwen2.5-3B-Instruct
  model_params_B            : 3.0
  dataset_name              : EleutherAI/asdiv
  dataset_split             : validation
  max_eval_samples          : 300
  random_seed               : 42
  n_votes                   : 5
  vote_temperature          : 0.4
  refiner_temperature       : 0.3
  max_new_tokens            : 350
  known_baseline_acc        : 53.7
  known_pipeline_acc        : 64.0
  known_baseline_B          : 7.5
  known_pipeline_B          : 10.5
  results_file              : /kaggle/working/asdiv_ceiling/results.jsonl
  report_file               : /kaggle/working/asdiv_ceiling/eval_report.json
  angle1_file               : /kaggle/working/asdiv_ceiling/angle1_compute_efficiency.json
  angle2_file               : /kaggle/working/asdiv_ceiling/angle2_vote_consistency.json
  angle3_file               : /kaggle/working/asdiv_ceiling/angle3_confidence_calibration.json
  angle4_file               : /kaggle/worki

In [5]:
# CELL 5 -- Load ASDiv dataset
# ASDiv fields: body, question, solution_type, answer, formula
# We combine body + question, strip units from answers (e.g. "10 cookies" -> "10"),
# and map solution_type to 5 broad categories for Angle 4.
import random

# ── Operation type mapping ──────────────────────────────────────────────
OP_MAP = {
    "addition"       : "Addition",
    "sum"            : "Addition",
    "subtraction"    : "Subtraction",
    "difference"     : "Subtraction",
    "comparison"     : "Subtraction",
    "multiplication" : "Multiplication",
    "division"       : "Division",
    "common-division": "Division",
    "floor-division" : "Division",
}

def broad_op(solution_type):
    st = str(solution_type).lower().strip()
    for key, val in OP_MAP.items():
        if key in st:
            return val
    return "Multi-step"

def normalise_asdiv(item):
    q = item["body"].strip().rstrip(".") + " " + item["question"].strip()
    raw_ans = str(item["answer"]).strip().replace(",", "")
    m = re.match(r"(-?[\d\.]+)", raw_ans)
    ans_str = m.group(1) if m else raw_ans
    try:
        f = float(ans_str)
        ans_str = str(int(f)) if f == int(f) else str(round(f, 4))
    except Exception:
        pass
    return {
        "question"      : q,
        "answer"        : ans_str,
        "op_type"       : broad_op(item.get("solution_type", "")),
        "solution_type" : str(item.get("solution_type", "")),
    }

print("Loading ASDiv from HuggingFace...")
raw_ds   = load_dataset(CONFIG["dataset_name"])
all_data = [normalise_asdiv(x) for x in raw_ds[CONFIG["dataset_split"]]]
print(f"Split size: {len(all_data)}")

op_counts = Counter(d["op_type"] for d in all_data)
print("Operation type distribution (full split):")
for op, cnt in sorted(op_counts.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {cnt}")

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])
n = min(CONFIG["max_eval_samples"], len(all_data))
test_data = random.sample(all_data, n)

sampled_op = Counter(d["op_type"] for d in test_data)
print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
print("Operation distribution in sample:")
for op, cnt in sorted(sampled_op.items(), key=lambda x: -x[1]):
    print(f"  {op:<20}: {cnt}")
print(f"\nFirst Q : {test_data[0]['question'][:80]}...")
print(f"First A : {test_data[0]['answer']}  |  op: {test_data[0]['op_type']}")

Loading ASDiv from HuggingFace...


README.md:   0%|          | 0.00/494 [00:00<?, ?B/s]

asdiv/validation-00000-of-00001.parquet:   0%|          | 0.00/267k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/2305 [00:00<?, ? examples/s]

Split size: 2305
Operation type distribution (full split):
  Multi-step          : 723
  Subtraction         : 568
  Addition            : 446
  Division            : 308
  Multiplication      : 260

Sampled 300 questions (seed=42)
Operation distribution in sample:
  Multi-step          : 89
  Subtraction         : 67
  Addition            : 61
  Multiplication      : 43
  Division            : 40

First Q : He also has a section filled with short story booklets. If each booklet has 9 pa...
First A : 441  |  op: Multiplication


In [6]:
# CELL 6 -- Answer extraction (identical to pipeline notebook)
def normalise_num(s):
    s = s.replace(",", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except ValueError:
        return s

def extract_gt_answer(answer_str):
    return normalise_num(str(answer_str))

def extract_pred_answer(text):
    m = re.search(r"####\s*(-?[\d\.]+)", text)
    if m: return normalise_num(m.group(1))
    m = re.search(r"\\boxed\{(-?[\d\.]+)\}", text)
    if m: return normalise_num(m.group(1))
    m = re.search(r"(?:the answer is|answer is)\s*:?\s*\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    m = re.search(r"=\s*\$?(-?[\d\.]+)\s*$", text.strip(), re.MULTILINE)
    if m: return normalise_num(m.group(1))
    m = re.search(r"\*\*\$?(-?[\d\.]+)\*\*\.?\s*$", text.strip())
    if m: return normalise_num(m.group(1))
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    return ""

_tests = [
    ("#### 42","42"),("#### 3.5","3.5"),("The answer is 7","7"),
    ("Total = 20","20"),("Therefore, 13","13"),("random text",""),
]
ok = all(extract_pred_answer(t)==e for t,e in _tests)
print("Extractor:", "ALL PASSED" if ok else "FAILURES DETECTED")
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    print(f"  {'OK' if got==exp else 'FAIL'}  '{txt}' -> '{got}'")

Extractor: ALL PASSED
  OK  '#### 42' -> '42'
  OK  '#### 3.5' -> '3.5'
  OK  'The answer is 7' -> '7'
  OK  'Total = 20' -> '20'
  OK  'Therefore, 13' -> '13'
  OK  'random text' -> ''


In [7]:
# CELL 7 -- Load 3B model (BASE only -- NO LoRA, NO fine-tuning)
print(f"Loading: {CONFIG['model_name']}")
print("Adapter: NONE -- base model only, no task-specific training")

model_tok = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if model_tok.pad_token is None:
    model_tok.pad_token = model_tok.eos_token

model_3b = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Model VRAM : {used:.2f} GB / {total:.1f} GB")
    print(f"Headroom   : {total - used:.1f} GB")

compute_per_q = CONFIG["model_params_B"] * CONFIG["n_votes"]
print(f"Compute per question: {CONFIG['model_params_B']}B x {CONFIG['n_votes']} = {compute_per_q}B param-passes")
print("Model ready -- no fine-tuning, no adapter")

Loading: Qwen/Qwen2.5-3B-Instruct
Adapter: NONE -- base model only, no task-specific training


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model VRAM : 5.75 GB / 15.9 GB
Headroom   : 10.1 GB
Compute per question: 3.0B x 5 = 15.0B param-passes
Model ready -- no fine-tuning, no adapter


In [8]:
# CELL 8 -- Generation functions
SOLVE_SYSTEM = (
    "You are a math problem solver.\n"
    "Compute each step numerically. No markdown. No bullet points. No headers.\n"
    "Write plain arithmetic steps only.\n"
    "Your absolute last line must be: #### [number]\n"
    "NEVER write ### or ** in your response.\n\n"
    "Example:\n"
    "Eaten = 14. Given = 13.\n"
    "Difference = 14 - 13 = 1.\n"
    "#### 1"
)

REFINER_SYSTEM = (
    "You are a careful math problem solver.\n"
    "Previous attempts on this problem gave different answers.\n"
    "Re-solve completely from scratch using a fresh approach.\n"
    "Show every arithmetic step.\n"
    "Your FINAL line must be exactly: #### [number]"
)

def run_3b(messages, max_tokens, temperature):
    prompt = model_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = model_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    dev    = next(model_3b.parameters()).device
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    with torch.no_grad():
        out = model_3b.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = model_tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return model_tok.decode(new_toks, skip_special_tokens=True).strip()

def generate_solve(question):
    return run_3b(
        [{"role":"system","content":SOLVE_SYSTEM},
         {"role":"user",  "content":f"Problem: {question}"}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )

def generate_refine(question, candidates):
    cands   = ", ".join(sorted(set(c for c in candidates if c)))
    content = f"Problem: {question}\n\nPrevious attempts disagreed: {cands}\nRe-solve carefully:"
    return run_3b(
        [{"role":"system","content":REFINER_SYSTEM},
         {"role":"user",  "content":content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )

print("Generation functions ready")
print(f"  generate_solve()   -- 3B base, temp {CONFIG['vote_temperature']}, no plan")
print(f"  generate_refine()  -- 3B base, temp {CONFIG['refiner_temperature']}, tie-breaker")

Generation functions ready
  generate_solve()   -- 3B base, temp 0.4, no plan
  generate_refine()  -- 3B base, temp 0.3, tie-breaker


In [9]:
# CELL 9 -- Voting logic (identical to pipeline notebook)
def vote_and_decide(answers, question, gt_answer=None):
    valid = [a for a in answers if a and a.strip()]
    if not valid: valid = answers
    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(answers)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / total
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])
    refiner_used = False; refiner_correct = None

    if is_majority:
        final = top_answer; strategy = "majority"
        conf  = round(top_count / total, 4); wasted = total - top_count
    else:
        ref_raw         = generate_refine(question, list(answers))
        ref_ans         = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None
        all_votes   = answers + [ref_ans]
        new_counts  = Counter(all_votes)
        new_common  = new_counts.most_common()
        new_top     = new_common[0][0]
        new_top_c   = new_common[0][1]
        still_tied  = len(new_common) > 1 and new_top_c == new_common[1][1]
        final       = new_top
        strategy    = "coin_flip" if still_tied else "refiner_tiebreak"
        conf        = round(new_top_c / len(all_votes), 4)
        vote_counts = new_counts; total = len(all_votes)
        correct_votes    = new_counts.get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / total
        wasted           = total - new_top_c

    return {
        "final_answer"    : final, "strategy":strategy, "confidence":conf,
        "vote_counts"     : dict(vote_counts), "correct_votes":correct_votes,
        "total_votes"     : total, "vote_consistency":round(vote_consistency,4),
        "wasted_votes"    : wasted, "refiner_used":refiner_used,
        "refiner_correct" : refiner_correct,
    }

print("Voting logic ready (majority / refiner_tiebreak / coin_flip)")

Voting logic ready (majority / refiner_tiebreak / coin_flip)


In [10]:
# CELL 10 -- Single question test
print("=" * 65)
print("SINGLE QUESTION TEST  (ASDiv -- 3B Ceiling)")
print("=" * 65)
item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
op   = item["op_type"]
print(f"Question  : {q}")
print(f"GT Answer : {gt}  |  Operation: {op}")
print(f"\nRunning {CONFIG['n_votes']} votes (3B base, no plan, temp={CONFIG['vote_temperature']})...")

votes_raw = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_solve(q)
    pred = extract_pred_answer(raw)
    votes_raw.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw[:80]: {raw[:80]}")

dec = vote_and_decide(votes_raw, q, gt)
print(f"\n  Result    : {dec['final_answer']}  (GT: {gt})  {'CORRECT' if dec['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {dec['strategy']}")
print(f"  Confidence: {dec['confidence']}")
print(f"  Correct votes: {dec['correct_votes']}/{dec['total_votes']}")
print(f"  Vote counts  : {dec['vote_counts']}")
print("\nTest done -- run Cell 11 for full 300-question evaluation")

SINGLE QUESTION TEST  (ASDiv -- 3B Ceiling)
Question  : He also has a section filled with short story booklets. If each booklet has 9 pages and there are 49 booklets in the short story section, how many pages will Jack need to go through if he plans to read them all?
GT Answer : 441  |  Operation: Multiplication

Running 5 votes (3B base, no plan, temp=0.4)...
  Vote 1: '441'  |  raw[:80]: 49times9=441.
#### [441]
  Vote 2: '441'  |  raw[:80]: 49times9=441.
#### [441]
  Vote 3: '441'  |  raw[:80]: 49times9=441.
#### [441]
  Vote 4: '441'  |  raw[:80]: TotalPages = 9 * 49 = 441.
#### [441]
  Vote 5: '441'  |  raw[:80]: TotalPages = 9 * 49 = 441.
#### [441]

  Result    : 441  (GT: 441)  CORRECT
  Strategy  : majority
  Confidence: 1.0
  Correct votes: 5/5
  Vote counts  : {'441': 5}

Test done -- run Cell 11 for full 300-question evaluation


In [11]:
# CELL 11 -- Full Evaluation Loop
# op_type is stored in every record so Angle 4 can group by operation type.

print(f"ASDiv 3B Ceiling evaluation: {len(test_data)} questions")
print(f"Model  : {CONFIG['model_name']} (base, no LoRA)")
print(f"Votes  : {CONFIG['n_votes']} x temp {CONFIG['vote_temperature']}")
print(f"Compute: {CONFIG['model_params_B']}B x {CONFIG['n_votes']} = {CONFIG['model_params_B']*CONFIG['n_votes']}B param-passes")
print("-" * 65)

results   = []
start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed from index {start_idx} ({len(results)} saved)")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="ASDiv 3B Ceiling"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])
    op_type   = item["op_type"]

    try:
        votes_raw = [extract_pred_answer(generate_solve(question))
                     for _ in range(CONFIG["n_votes"])]
        dec = vote_and_decide(votes_raw, question, gt_answer)
        results.append({
            "mode"            : "ceiling_3b",
            "idx"             : idx,
            "question"        : question,
            "gt_answer"       : gt_answer,
            "op_type"         : op_type,
            "final_answer"    : dec["final_answer"],
            "correct"         : dec["final_answer"] == gt_answer,
            "strategy"        : dec["strategy"],
            "confidence"      : dec["confidence"],
            "correct_votes"   : dec["correct_votes"],
            "total_votes"     : dec["total_votes"],
            "vote_consistency": dec["vote_consistency"],
            "wasted_votes"    : dec["wasted_votes"],
            "refiner_used"    : dec["refiner_used"],
            "refiner_correct" : dec["refiner_correct"],
            "vote_counts"     : dec["vote_counts"],
        })
    except RuntimeError as e:
        results.append({
            "mode":"ceiling_3b","idx":idx,"question":question,
            "gt_answer":gt_answer,"op_type":op_type,
            "final_answer":"","correct":False,"strategy":"error",
            "confidence":0.0,"correct_votes":0,"total_votes":CONFIG["n_votes"],
            "vote_consistency":0.0,"wasted_votes":CONFIG["n_votes"],
            "refiner_used":False,"refiner_correct":None,"vote_counts":{},"error":str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"],"w") as f:
            for r in results: f.write(json.dumps(r)+"\n")
        with open(CONFIG["checkpoint_file"],"w") as f:
            json.dump({"last_index":idx+1},f)
        acc  = sum(r["correct"] for r in results)/len(results)*100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:3d}/{len(test_data)}]  3B Ceiling: {acc:.1f}%  ({mins:.1f} min)")

with open(CONFIG["results_file"],"w") as f:
    for r in results: f.write(json.dumps(r)+"\n")
with open(CONFIG["checkpoint_file"],"w") as f:
    json.dump({"last_index":len(test_data)},f)

correct = sum(r["correct"] for r in results)
acc     = correct/len(results)*100
print(f"\nEvaluation complete.")
print(f"  3B Ceiling : {correct}/{len(results)} = {acc:.1f}%")
print(f"  Pipeline   : {CONFIG['known_pipeline_acc']}%  (known, 10.5B)")
print(f"  Baseline   : {CONFIG['known_baseline_acc']}%  (known, 7.5B)")

ASDiv 3B Ceiling evaluation: 300 questions
Model  : Qwen/Qwen2.5-3B-Instruct (base, no LoRA)
Votes  : 5 x temp 0.4
Compute: 3.0B x 5 = 15.0B param-passes
-----------------------------------------------------------------
Starting fresh


ASDiv 3B Ceiling:   0%|          | 0/300 [00:00<?, ?it/s]

  [ 25/300]  3B Ceiling: 28.0%  (3.4 min)
  [ 50/300]  3B Ceiling: 24.0%  (7.9 min)
  [ 75/300]  3B Ceiling: 28.0%  (12.0 min)
  [100/300]  3B Ceiling: 26.0%  (15.9 min)
  [125/300]  3B Ceiling: 26.4%  (20.3 min)
  [150/300]  3B Ceiling: 26.7%  (24.6 min)
  [175/300]  3B Ceiling: 26.9%  (28.6 min)
  [200/300]  3B Ceiling: 26.5%  (32.0 min)
  [225/300]  3B Ceiling: 27.6%  (34.6 min)
  [250/300]  3B Ceiling: 27.6%  (38.7 min)
  [275/300]  3B Ceiling: 28.0%  (43.0 min)
  [300/300]  3B Ceiling: 28.7%  (48.0 min)

Evaluation complete.
  3B Ceiling : 86/300 = 28.7%
  Pipeline   : 64.0%  (known, 10.5B)
  Baseline   : 53.7%  (known, 7.5B)


In [12]:
# CELL 12 -- ANGLE 1: COMPUTE EFFICIENCY
ceiling_compute  = CONFIG["model_params_B"] * CONFIG["n_votes"]   # 15.0
pipeline_compute = CONFIG["known_pipeline_B"]                       # 10.5
baseline_compute = CONFIG["known_baseline_B"]                       # 7.5

ceiling_acc  = sum(r["correct"] for r in results)/len(results)*100
pipeline_acc = CONFIG["known_pipeline_acc"]
baseline_acc = CONFIG["known_baseline_acc"]

ceiling_eff  = ceiling_acc  / ceiling_compute
pipeline_eff = pipeline_acc / pipeline_compute
baseline_eff = baseline_acc / baseline_compute

n = len(results); N = CONFIG["n_votes"]
ceiling_wasted = sum(r["wasted_votes"] for r in results)
strategy_stats = {}
for r in results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

ref_triggered = sum(r["refiner_used"] for r in results)
ref_correct   = sum(1 for r in results if r["refiner_used"] and r.get("refiner_correct"))

print("=" * 68)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (ASDiv -- Three-Way Comparison)")
print("=" * 68)
print(f"  {'Setup':<38} | {'Compute':>8} | {'Accuracy':>9} | {'Acc/B':>7}")
print(f"  {'-'*38}-+-{'-'*8}-+-{'-'*9}-+-{'-'*7}")
print(f"  {'Baseline  (1.5B x 5)':<38} | {baseline_compute:>6.1f}B  | {baseline_acc:>8.1f}% | {baseline_eff:>6.3f}")
print(f"  {'Pipeline  (3B+LoRA x1 + 1.5B x5)':<38} | {pipeline_compute:>6.1f}B  | {pipeline_acc:>8.1f}% | {pipeline_eff:>6.3f}")
print(f"  {'Ceiling   (3B base x 5)  <- THIS':<38} | {ceiling_compute:>6.1f}B  | {ceiling_acc:>8.1f}% | {ceiling_eff:>6.3f}")
print()
print(f"  Pipeline vs Ceiling  : {pipeline_acc - ceiling_acc:+.1f} pts  (pipeline uses {ceiling_compute - pipeline_compute:.1f}B LESS)")
print(f"  Ceiling  vs Baseline : {ceiling_acc  - baseline_acc:+.1f} pts  (+{ceiling_compute - baseline_compute:.1f}B more)")
print(f"  Pipeline vs Baseline : {pipeline_acc - baseline_acc:+.1f} pts  (+{pipeline_compute - baseline_compute:.1f}B more)")
print()
if pipeline_acc >= ceiling_acc:
    print(f"  RESULT: Pipeline MATCHES or BEATS ceiling at 30% lower cost.")
    print(f"  Fine-tuned structured planning is more efficient than raw capacity scaling.")
else:
    gap  = ceiling_acc - pipeline_acc
    frac = (pipeline_acc - baseline_acc) / max(ceiling_acc - baseline_acc, 0.01) * 100
    print(f"  RESULT: 3B ceiling leads pipeline by {gap:.1f} pts at 43% higher cost.")
    print(f"  Pipeline recovers {frac:.0f}% of ceiling gain at {pipeline_compute/ceiling_compute*100:.0f}% of ceiling cost.")

print(f"\n  Wasted votes -- Ceiling: {ceiling_wasted}/{n*N} ({ceiling_wasted/(n*N)*100:.1f}%)")
if ref_triggered > 0:
    print(f"  Refiner: triggered {ref_triggered}, correct {ref_correct}")
print(f"\n  Strategy breakdown:")
for s,v in sorted(strategy_stats.items(), key=lambda x:-x[1]['n']):
    acc_s = v['correct']/v['n']*100 if v['n'] else 0
    print(f"    {s:<22}  n={v['n']:3d}  acc={acc_s:.1f}%")

angle1 = {
    "dataset":"ASDiv","experiment":"ceiling_3b","n_questions":n,
    "ceiling_accuracy":round(ceiling_acc,2),"pipeline_accuracy":pipeline_acc,"baseline_accuracy":baseline_acc,
    "ceiling_compute_B":ceiling_compute,"pipeline_compute_B":pipeline_compute,"baseline_compute_B":baseline_compute,
    "ceiling_efficiency":round(ceiling_eff,4),"pipeline_efficiency":round(pipeline_eff,4),"baseline_efficiency":round(baseline_eff,4),
    "ceiling_wasted_votes":ceiling_wasted,"refiner_triggered":ref_triggered,"refiner_correct":ref_correct,
    "strategy_breakdown":strategy_stats,"pipeline_beats_ceiling":pipeline_acc >= ceiling_acc,
}
with open(CONFIG["angle1_file"],"w") as f: json.dump(angle1,f,indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")

ANGLE 1 -- COMPUTE EFFICIENCY  (ASDiv -- Three-Way Comparison)
  Setup                                  |  Compute |  Accuracy |   Acc/B
  ---------------------------------------+----------+-----------+--------
  Baseline  (1.5B x 5)                   |    7.5B  |     53.7% |  7.160
  Pipeline  (3B+LoRA x1 + 1.5B x5)       |   10.5B  |     64.0% |  6.095
  Ceiling   (3B base x 5)  <- THIS       |   15.0B  |     28.7% |  1.911

  Pipeline vs Ceiling  : +35.3 pts  (pipeline uses 4.5B LESS)
  Ceiling  vs Baseline : -25.0 pts  (+7.5B more)
  Pipeline vs Baseline : +10.3 pts  (+3.0B more)

  RESULT: Pipeline MATCHES or BEATS ceiling at 30% lower cost.
  Fine-tuned structured planning is more efficient than raw capacity scaling.

  Wasted votes -- Ceiling: 412/1500 (27.5%)
  Refiner: triggered 33, correct 19

  Strategy breakdown:
    majority                n=267  acc=30.3%
    refiner_tiebreak        n= 23  acc=17.4%
    coin_flip               n= 10  acc=10.0%

Saved -> /kaggle/working/as

In [13]:
# CELL 13 -- ANGLE 2: VOTE CONSISTENCY
cons_scores = [r["vote_consistency"] for r in results]
mean_cons   = np.mean(cons_scores)

def bucket(scores):
    return {
        "all_wrong  (0%)"  : sum(1 for s in scores if s == 0.0),
        "low       (1-39%)": sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)" : sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)" : sum(1 for s in scores if s >= 0.8),
    }

dist      = bucket(cons_scores)
corr_cons = [r["vote_consistency"] for r in results if r["correct"]]

# Known values from N=300 pipeline run
pipeline_cons = 0.586   # 58.6% from ASDiv N=300
baseline_cons = 0.496   # 49.6% from ASDiv N=300
pipeline_dist = {"all_wrong  (0%)":49,"low       (1-39%)":33,"medium  (40-79%)":60,"high   (80-100%)":158}

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (ASDiv -- Three-Way)")
print("=" * 65)
print(f"  Mean correct-vote ratio (out of {CONFIG['n_votes']} per question):")
print(f"    Baseline (1.5B x5) : {baseline_cons*100:.1f}%  ({baseline_cons*5:.2f}/5 avg)")
print(f"    Pipeline (3B+LoRA) : {pipeline_cons*100:.1f}%  ({pipeline_cons*5:.2f}/5 avg)")
print(f"    Ceiling  (3B base) : {mean_cons*100:.1f}%  ({mean_cons*5:.2f}/5 avg)  <- THIS")

print(f"\n  Distribution (3B Ceiling vs Pipeline):")
print(f"  {'Bucket':<22} | {'Ceiling':>8} | {'Pipeline':>8}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}")
for bkt in ["all_wrong  (0%)","low       (1-39%)","medium  (40-79%)","high   (80-100%)"]:
    cv = dist[bkt]; pv = pipeline_dist.get(bkt,"—")
    print(f"  {bkt:<22} | {cv:>8} | {pv:>8}")

if corr_cons:
    print(f"\n  Correct questions: ceiling consistency = {np.mean(corr_cons)*100:.1f}% (n={len(corr_cons)})")

angle2 = {
    "dataset":"ASDiv","experiment":"ceiling_3b","n_questions":len(results),
    "ceiling_mean_consistency":round(mean_cons,4),
    "pipeline_mean_consistency":pipeline_cons,"baseline_mean_consistency":baseline_cons,
    "ceiling_distribution":dist,
    "ceiling_correct_q_consistency":round(np.mean(corr_cons),4) if corr_cons else 0,
}
with open(CONFIG["angle2_file"],"w") as f: json.dump(angle2,f,indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")

ANGLE 2 -- VOTE CONSISTENCY  (ASDiv -- Three-Way)
  Mean correct-vote ratio (out of 5 per question):
    Baseline (1.5B x5) : 49.6%  (2.48/5 avg)
    Pipeline (3B+LoRA) : 58.6%  (2.93/5 avg)
    Ceiling  (3B base) : 24.1%  (1.20/5 avg)  <- THIS

  Distribution (3B Ceiling vs Pipeline):
  Bucket                 |  Ceiling | Pipeline
  -----------------------+----------+---------
  all_wrong  (0%)        |      173 |       49
  low       (1-39%)      |       47 |       33
  medium  (40-79%)       |       28 |       60
  high   (80-100%)       |       52 |      158

  Correct questions: ceiling consistency = 73.2% (n=86)

Saved -> /kaggle/working/asdiv_ceiling/angle2_vote_consistency.json


In [14]:
# CELL 14 -- ANGLE 3: CONFIDENCE CALIBRATION
def calibration_report(res, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40,  0.25),
    ]
    n_total = len(res); ece = 0.0; calib_out = []
    false_conf = sum(1 for r in res if r["confidence"] >= 0.80 and not r["correct"])
    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in res if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |"); continue
        n=len(subset); acc=sum(r["correct"] for r in subset)/n; gap=abs(acc-mid)
        ece += (n/n_total)*gap; flag="Good" if gap<0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),"expected":mid,"gap":round(gap,4)})
    hc = [r for r in res if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc)/max(1,len(hc))*100
    print(f"  {'ECE (lower=better)':<26}   {ece:.4f}")
    print(f"  High-conf questions : {len(hc)}  |  Accuracy when confident: {hc_acc:.1f}%")
    print(f"  Confidently WRONG   : {false_conf}")
    return ece, calib_out, false_conf

# Known ECE from N=300 pipeline run
known_pipe_ece = 0.1002
known_base_ece = 0.1827

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (ASDiv -- 3B Ceiling)")
print("=" * 65)
ceiling_ece, ceiling_calib, ceiling_false = calibration_report(results, "3B CEILING (this run)")
print(f"\n  ECE -- Three-Way:")
print(f"    Baseline (1.5B x5) : {known_base_ece:.4f}")
print(f"    Pipeline (3B+LoRA) : {known_pipe_ece:.4f}")
print(f"    Ceiling  (3B base) : {ceiling_ece:.4f}  <- THIS")
print(f"\n  Ceiling vs Pipeline: {ceiling_ece - known_pipe_ece:+.4f} ({'worse' if ceiling_ece > known_pipe_ece else 'better'})")
print(f"  False confidence (ceiling): {ceiling_false}")

angle3 = {
    "dataset":"ASDiv","experiment":"ceiling_3b","n_questions":len(results),
    "ceiling_ece":round(ceiling_ece,4),"pipeline_ece":known_pipe_ece,"baseline_ece":known_base_ece,
    "ceiling_false_confidence":ceiling_false,"ceiling_calibration":ceiling_calib,
}
with open(CONFIG["angle3_file"],"w") as f: json.dump(angle3,f,indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")

ANGLE 3 -- CONFIDENCE CALIBRATION  (ASDiv -- 3B Ceiling)

  [3B CEILING (this run)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |   176 |     29.5% |       90% |  0.605 | Poor
  High       (0.60-0.80)     |    37 |     32.4% |       70% |  0.376 | Poor
  Medium     (0.40-0.60)     |    49 |     24.5% |       50% |  0.255 | Poor
  Low        (<0.40)         |    38 |     26.3% |       25% |  0.013 | Good
  ECE (lower=better)           0.4443
  High-conf questions : 176  |  Accuracy when confident: 29.5%
  Confidently WRONG   : 124

  ECE -- Three-Way:
    Baseline (1.5B x5) : 0.1827
    Pipeline (3B+LoRA) : 0.1002
    Ceiling  (3B base) : 0.4443  <- THIS

  Ceiling vs Pipeline: +0.3441 (worse)
  False confidence (ceiling): 124

Saved -> /kaggle/working/asdiv_ceiling/angle3_confidence_calibration.json


In [15]:
# CELL 15 -- ANGLE 4: ACCURACY BY OPERATION TYPE  (ASDiv exclusive)
# =================================================================
# Breaks results down by: Addition / Subtraction / Multiplication /
# Division / Multi-step.
#
# For each op type we compare 3B ceiling vs our pipeline (known) vs baseline (known).
# Key question: does the 3B base model match or beat the pipeline on specific op types?
# =================================================================

ceiling_by_op = defaultdict(list)
for r in results:
    ceiling_by_op[r.get("op_type","Unknown")].append(r)

# Known pipeline values by op type (N=300 run)
known_pipe = CONFIG["known_pipeline_by_op"]

all_ops = sorted(set(list(ceiling_by_op.keys()) + list(known_pipe.keys())))

print("=" * 80)
print("ANGLE 4 -- ACCURACY BY OPERATION TYPE  (ASDiv -- 3B Ceiling vs Pipeline)")
print("=" * 80)
print(f"  {'Operation':<16} | {'N':>4} | {'Ceiling':>8} | {'Pipeline':>8} | {'Baseline':>9} | {'C vs P':>7} | {'C vs B':>7}")
print(f"  {'-'*16}-+-{'-'*4}-+-{'-'*8}-+-{'-'*8}-+-{'-'*9}-+-{'-'*7}-+-{'-'*7}")

op_results = {}
for op in all_ops:
    c_items = ceiling_by_op.get(op, [])
    n = len(c_items)
    if n == 0: continue

    c_acc  = sum(r["correct"] for r in c_items) / n * 100
    p_info = known_pipe.get(op, {})
    p_acc  = p_info.get("guided",  0.0)
    b_acc  = p_info.get("baseline", 0.0)

    c_vs_p = c_acc - p_acc
    c_vs_b = c_acc - b_acc
    sign_p = "+" if c_vs_p >= 0 else ""
    sign_b = "+" if c_vs_b >= 0 else ""

    print(f"  {op:<16} | {n:>4} | {c_acc:>7.1f}% | {p_acc:>7.1f}% | {b_acc:>8.1f}% | {sign_p+str(round(c_vs_p,1))+'%':>7} | {sign_b+str(round(c_vs_b,1))+'%':>7}")
    op_results[op] = {
        "n"              : n,
        "ceiling_acc"    : round(c_acc, 2),
        "pipeline_acc"   : p_acc,
        "baseline_acc"   : b_acc,
        "ceiling_vs_pipeline" : round(c_vs_p, 2),
        "ceiling_vs_baseline" : round(c_vs_b, 2),
    }

print()
# Multi-step finding reminder
multi = op_results.get("Multi-step")
if multi:
    if multi["ceiling_acc"] > multi["pipeline_acc"]:
        print(f"  NOTE: 3B ceiling leads pipeline on Multi-step ({multi['ceiling_acc']:.1f}% vs {multi['pipeline_acc']:.1f}%)")
        print(f"        Raw capacity helps more than planning on compound problem chains.")
    else:
        print(f"  NOTE: Pipeline matches/beats 3B ceiling on Multi-step ({multi['pipeline_acc']:.1f}% vs {multi['ceiling_acc']:.1f}%)")

angle4 = {
    "dataset":"ASDiv","experiment":"ceiling_3b",
    "n_questions":len(results),"by_operation":op_results,
}
with open(CONFIG["angle4_file"],"w") as f: json.dump(angle4,f,indent=2)
print(f"\nSaved -> {CONFIG['angle4_file']}")

ANGLE 4 -- ACCURACY BY OPERATION TYPE  (ASDiv -- 3B Ceiling vs Pipeline)
  Operation        |    N |  Ceiling | Pipeline |  Baseline |  C vs P |  C vs B
  -----------------+------+----------+----------+-----------+---------+--------
  Addition         |   61 |    42.6% |    73.8% |     55.7% |  -31.2% |  -13.1%
  Division         |   40 |    10.0% |    60.0% |     40.0% |  -50.0% |  -30.0%
  Multi-step       |   89 |    21.3% |    53.9% |     53.9% |  -32.6% |  -32.6%
  Multiplication   |   43 |    37.2% |    72.1% |     55.8% |  -34.9% |  -18.6%
  Subtraction      |   67 |    31.3% |    64.2% |     56.7% |  -32.9% |  -25.4%

  NOTE: Pipeline matches/beats 3B ceiling on Multi-step (53.9% vs 21.4%)

Saved -> /kaggle/working/asdiv_ceiling/angle4_by_operation_type.json


In [16]:
# CELL 16 -- Full Three-Way Summary Table
with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)
with open(CONFIG["angle4_file"]) as f: a4 = json.load(f)

ca = a1["ceiling_accuracy"]
pa = a1["pipeline_accuracy"]
ba = a1["baseline_accuracy"]

print("=" * 75)
print("  ASDiv -- THREE-WAY: BASELINE / PIPELINE / 3B CEILING")
print(f"  N={a1['n_questions']} questions  |  Seed={CONFIG['random_seed']}  |  Qwen2.5 model family")
print("=" * 75)

rows = [
    ["Metric",                  "Baseline",          "Our Pipeline",          "3B Ceiling (this)"],
    ["Model",                   "Qwen2.5-1.5B x5",   "3B LoRA + 1.5B x5",    "Qwen2.5-3B base x5"],
    ["Fine-Tuning",             "None",              "LoRA on GSM8K",         "None"],
    ["Compute (param-passes)",  "7.5B",              "10.5B",                 "15.0B"],
    ["─────────────────────",   "───────────────",   "───────────────────",   "──────────────────"],
    ["Overall Accuracy",        f"{ba:.1f}%",         f"{pa:.1f}%",            f"{ca:.1f}%"],
    ["vs Baseline",             "—",                 f"+{pa-ba:.1f} pts",      f"+{ca-ba:.1f} pts"],
    ["Pipeline vs Ceiling",     "—",                 f"{'WINS' if pa>=ca else 'loses'} {abs(pa-ca):.1f} pts","←"],
    ["Acc / Billion passes",    f"{ba/7.5:.3f}",      f"{pa/10.5:.3f}",        f"{ca/15.0:.3f}"],
    ["─────────────────────",   "───────────────",   "───────────────────",   "──────────────────"],
    ["Vote Consistency",        f"{a2['baseline_mean_consistency']*100:.1f}%",
                                 f"{a2['pipeline_mean_consistency']*100:.1f}%",
                                 f"{a2['ceiling_mean_consistency']*100:.1f}%"],
    ["ECE (lower=better)",      f"{a3['baseline_ece']:.4f}",
                                 f"{a3['pipeline_ece']:.4f}",
                                 f"{a3['ceiling_ece']:.4f}"],
    ["False Confidence",        "—",                 "—",                     str(a3["ceiling_false_confidence"])],
]

col_w = [26, 20, 22, 20]
sep   = "-+-".join("-"*w for w in col_w)
for i, row in enumerate(rows):
    if "─────" in row[0]:
        print("  " + sep); continue
    line = " | ".join(str(c).ljust(col_w[j]) for j,c in enumerate(row))
    print("  " + line)
    if i == 0: print("  " + sep)

print()
print("  Operation-Type Breakdown (Ceiling vs Pipeline):")
for op, v in sorted(a4["by_operation"].items(), key=lambda x: x[1].get("n",0), reverse=True):
    c_vs_p = v["ceiling_vs_pipeline"]
    sign   = "↑ ceiling wins" if c_vs_p > 0 else ("tie" if c_vs_p == 0 else "↓ pipeline wins")
    print(f"    {op:<16}  n={v['n']:3d}  ceiling={v['ceiling_acc']:.1f}%  pipeline={v['pipeline_acc']:.1f}%  ({c_vs_p:+.1f} pts  {sign})")

print()
print("=" * 75)
if pa >= ca:
    print(f"  VERDICT: Pipeline BEATS the 3B ceiling (+{pa-ca:.1f} pts) at 30% lower cost.")
    print(f"  Fine-tuned structured planning > raw model capacity scaling on ASDiv.")
else:
    gap  = ca - pa
    frac = (pa - ba) / max(ca - ba, 0.01) * 100
    print(f"  VERDICT: 3B ceiling leads pipeline by {gap:.1f} pts.")
    print(f"  Pipeline delivers {frac:.0f}% of ceiling gain at {pa/ca*100:.0f}% of ceiling cost.")
    print(f"  See operation-type breakdown above for where each condition is strongest.")
print("=" * 75)

full = {
    "dataset":"ASDiv","seed":CONFIG["random_seed"],"n_questions":a1["n_questions"],
    "conditions":{
        "baseline":{"compute_B":7.5, "accuracy":ba,"model":"Qwen2.5-1.5B x5","fine_tuned":False},
        "pipeline":{"compute_B":10.5,"accuracy":pa,"model":"3B LoRA + 1.5B x5","fine_tuned":True},
        "ceiling" :{"compute_B":15.0,"accuracy":ca,"model":"3B base x5","fine_tuned":False},
    },
    "pipeline_beats_ceiling":pa >= ca,
    "angle1":a1,"angle2":a2,"angle3":a3,"angle4":a4,
}
with open(CONFIG["report_file"],"w") as f: json.dump(full,f,indent=2)
print(f"\nAll results saved to {OUTPUT_DIR}/")
print("Files: results.jsonl · eval_report.json · angle1/2/3/4.json")

  ASDiv -- THREE-WAY: BASELINE / PIPELINE / 3B CEILING
  N=300 questions  |  Seed=42  |  Qwen2.5 model family
  Metric                     | Baseline             | Our Pipeline           | 3B Ceiling (this)   
  ---------------------------+----------------------+------------------------+---------------------
  Model                      | Qwen2.5-1.5B x5      | 3B LoRA + 1.5B x5      | Qwen2.5-3B base x5  
  Fine-Tuning                | None                 | LoRA on GSM8K          | None                
  Compute (param-passes)     | 7.5B                 | 10.5B                  | 15.0B               
  ---------------------------+----------------------+------------------------+---------------------
  Overall Accuracy           | 53.7%                | 64.0%                  | 28.7%               
  vs Baseline                | —                    | +10.3 pts              | +-25.0 pts          
  Pipeline vs Ceiling        | —                    | WINS 35.3 pts          | ←          